####  Run this cell to set up and start your interactive session.


In [2]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

You are already connected to a glueetl session e637331a-86b0-4532-bba4-a28be287cd6b.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.


You are already connected to a glueetl session e637331a-86b0-4532-bba4-a28be287cd6b.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 5.0


You are already connected to a glueetl session e637331a-86b0-4532-bba4-a28be287cd6b.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: None
Setting new worker type to: G.1X


You are already connected to a glueetl session e637331a-86b0-4532-bba4-a28be287cd6b.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: None
Setting new number of workers to: 5



In [5]:
from awsglue.dynamicframe import DynamicFrame
from pyspark.sql.functions import current_timestamp, date_format
from pyspark.sql.functions import col, current_timestamp, to_date, explode
from datetime import datetime

In [11]:
# --- S3 paths ---
BUCKET = "bitcoindatapipelineproject"
RAW_PATH = f"s3://{BUCKET}/raw_layer/unprocessed_raw/"               # input JSON files
OUTPUT_PATH = f"s3://{BUCKET}/refined_layer"                            # output base

# --- Read raw JSON using spark.read (not DynamicFrame) ---
# If files are JSON arrays, multiLine=True will parse correctly.
raw_df = (
    spark.read
         .option("multiLine", True)
         .json(RAW_PATH)
)

In [13]:
raw_df.count()

50


In [14]:
raw_df.head()

Row(ath=124128.0, ath_change_percentage=-5.13803, ath_date='2025-08-14T00:37:02.582Z', atl=67.81, atl_change_percentage=173549.97256, atl_date='2013-07-06T00:00:00.000Z', circulating_supply=19907787.0, current_price=117716.0, fully_diluted_valuation=2343808258414, high_24h=118514.0, id='bitcoin', image='https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400', last_updated='2025-08-17T21:15:11.723Z', low_24h=117279.0, market_cap=2343808258414, market_cap_change_24h=1674591358.0, market_cap_change_percentage_24h=0.0715, market_cap_rank=1, max_supply=21000000.0, name='Bitcoin', price_change_24h=70.23, price_change_percentage_24h=0.0597, roi=None, symbol='btc', total_supply=19907787.0, total_volume=20761670348)


In [15]:
# ──────────────────────────────────────────────────────────────────────────────
# Processors (match your coin schema from Lambda version: flat keys on the coin)
# ──────────────────────────────────────────────────────────────────────────────
def process_metadata(df):
    """
    Produce metadata table:
      coin_id, symbol, name, image_url, market_cap_rank, max_supply, extracted_at
    """
    meta = (
        df.select(
            col("id").alias("coin_id"),
            col("symbol"),
            col("name"),
            col("image").alias("image_url"),
            col("market_cap_rank"),
            col("max_supply")
        )
        .dropDuplicates(["coin_id"])
        .withColumn("extracted_at", current_timestamp())
    )
    return meta

def process_market(df):
    """
    Produce market table (flat fields like your Lambda version):
      coin_id, price_usd, market_cap_usd, volume_usd, high_24h_usd, low_24h_usd,
      price_change_24h, price_change_pct_24h, market_cap_change_24h, market_cap_change_pct_24h, extracted_at
    """
    mkt = (
        df.select(
            col("id").alias("coin_id"),
            col("current_price").alias("price_usd"),
            col("market_cap").alias("market_cap_usd"),
            col("total_volume").alias("volume_usd"),
            col("high_24h").alias("high_24h_usd"),
            col("low_24h").alias("low_24h_usd"),
            col("price_change_24h"),
            col("price_change_percentage_24h").alias("price_change_pct_24h"),
            col("market_cap_change_24h"),
            col("market_cap_change_percentage_24h").alias("market_cap_change_pct_24h")
        )
        .dropDuplicates(["coin_id"])
        .withColumn("extracted_at", current_timestamp())
    )
    return mkt



In [17]:
# ──────────────────────────────────────────────────────────────────────────────
# Create tables
# ──────────────────────────────────────────────────────────────────────────────
metadata_df = process_metadata(raw_df)
market_df   = process_market(raw_df)



In [18]:
market_df.head()

Row(coin_id='staked-ether', price_usd=4463.19, market_cap_usd=39396283472, volume_usd=64460975, high_24h_usd=4554.34, low_24h_usd=4393.8, price_change_24h=45.19, price_change_pct_24h=1.02286, market_cap_change_24h=335865023.0, market_cap_change_pct_24h=0.85986, extracted_at=datetime.datetime(2025, 8, 18, 0, 31, 0, 425000))


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Writer (convert to DynamicFrame for Glue writer, like your Spotify example)
# ──────────────────────────────────────────────────────────────────────────────
# def write_to_s3(df, path_suffix, fmt="csv"):
#     dyf = DynamicFrame.fromDF(df.coalesce(1), glueContext, "dyf_out")
#     glueContext.write_dynamic_frame.from_options(
#         frame=dyf,
#         connection_type="s3",
#         connection_options={"path": f"{OUT_BASE}/{path_suffix}/"},
#         format=fmt,
#         format_options={"withHeader": True}
#     )

# Timestamped folders (daily-style like your Spotify script)
# ts = datetime.now().strftime("%Y-%m-%d")

# # ──────────────────────────────────────────────────────────────────────────────
# # Write outputs
# # ──────────────────────────────────────────────────────────────────────────────
# write_to_s3(metadata_df, f"meta_data/meta_data_transformed_{ts}", "csv")
# write_to_s3(market_df,   f"market_data/market_data_transformed_{ts}", "csv")

# job.commit()

In [ ]:
    # df_ts = (df
    #     .withColumn("ingest_ts", current_timestamp())                 # precise write time (UTC)
    #     .withColumn("run_id", date_format(current_timestamp(), "yyyyMMdd_HHmmss")))  # 

In [21]:
from awsglue.dynamicframe import DynamicFrame 
from datetime import datetime # --- write helper (simple + consistent with raw read) --- 

In [22]:

def write_to_s3(df, path, fmt="csv"): 
    df = df.coalesce(1) # Convert to DynamicFrame 
    dyf = DynamicFrame.fromDF(df, glueContext, "dyf") 
    glueContext.write_dynamic_frame.from_options( 
        frame=dyf, 
        connection_type="s3", 
        connection_options={"path": path},
        format=fmt,
        format_options={"withHeader": True} ) # --- refined outputs --- 


In [23]:
write_to_s3(metadata_df, "s3://bitcoindatapipelineproject/refined_layer/meta_data/meta_data_transformed_{}".format(datetime.now().strftime("%Y-%m-%d")), "csv") 
write_to_s3(market_df, "s3://bitcoindatapipelineproject/refined_layer/market_data/market_data_transformed_{}".format(datetime.now().strftime("%Y-%m-%d")), "csv") 
job.commit()